<a href="https://colab.research.google.com/github/ywoo-create/osspteamproject/blob/uirin/%EC%A0%84%EC%B2%98%EB%A6%AC%EB%B0%8F%EC%A6%9D%EA%B0%95.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install librosa soundfile numpy noisereduce -q

In [2]:
import os

GITHUB_USER = "ywoo-create"
GITHUB_REPO = "osspteamproject"
GITHUB_BRANCH = "dataset"

zip_url = f"https://github.com/{GITHUB_USER}/{GITHUB_REPO}/archive/refs/heads/{GITHUB_BRANCH}.zip"

!wget -q -O dataset.zip "{zip_url}"
!unzip -q -o dataset.zip
!rm dataset.zip

# 압축 풀리는 폴더명은 보통 "레포이름-브랜치명" 형식
EXTRACTED_DIR = f"{GITHUB_REPO}-{GITHUB_BRANCH}"
print("압축 해제 완료. 폴더 목록:")
!ls "{EXTRACTED_DIR}"

압축 해제 완료. 폴더 목록:
 0_indoor_alarms      2_emergency_alarms   시범용.m4a  '보이스톡 전화 소리.wav'
 1_outdoor_warnings   전혀다른.m4a	   test


In [3]:
import numpy as np
import librosa
import soundfile as sf
import random
import shutil

SAMPLE_RATE = 22050  # 모든 파일을 이 샘플링레이트로 통일해서 불러옴

SOURCE_ROOT = EXTRACTED_DIR          # 원본 데이터가 있는 루트
OUTPUT_ROOT = "augmented_dataset"    # 최종 증강 결과를 저장할 루트

CLASS_DIRS = [
    "0_indoor_alarms",
    "1_outdoor_warnings",
    "2_emergency_alarms",
]

os.makedirs(OUTPUT_ROOT, exist_ok=True)
for c in CLASS_DIRS:
    os.makedirs(os.path.join(OUTPUT_ROOT, c), exist_ok=True)

In [4]:
import noisereduce as nr

def reduce_noise(y, sr=SAMPLE_RATE):
    """스펙트럴 게이팅 기반 노이즈 제거. 잡음 구간을 따로 지정하지 않아도
    오디오 전체에서 잡음 프로파일을 통계적으로 추정해 제거합니다."""
    try:
        y_denoised = nr.reduce_noise(y=y, sr=sr, stationary=False)
    except Exception as e:
        print(f"  [경고] 노이즈 제거 실패, 원본 사용: {e}")
        y_denoised = y
    return y_denoised

def trim_silence(y, top_db=30):
    """앞뒤에 붙은 무음(또는 거의 무음인) 구간을 제거합니다.
    top_db가 낮을수록 더 작은 소리도 '소리 있음'으로 인식해 덜 자릅니다."""
    y_trimmed, _ = librosa.effects.trim(y, top_db=top_db)
    if len(y_trimmed) == 0:
        # 전부 잘려나가는 경우를 방지하는 안전장치
        return y
    return y_trimmed

def peak_normalize(y, target_peak=0.95):
    """피크 기준으로 음량을 정규화해서 파일 간 음량 차이를 통일합니다."""
    max_val = np.max(np.abs(y))
    if max_val > 0:
        y = y / max_val * target_peak
    return y

def preprocess_audio(y, sr=SAMPLE_RATE):
    """전처리 파이프라인: 노이즈 제거 -> 무음 트리밍 -> 정규화"""
    y = reduce_noise(y, sr)
    y = trim_silence(y)
    y = peak_normalize(y)
    return y

In [5]:
PREPROCESSED_ROOT = "preprocessed_dataset"
os.makedirs(PREPROCESSED_ROOT, exist_ok=True)
for c in CLASS_DIRS:
    os.makedirs(os.path.join(PREPROCESSED_ROOT, c), exist_ok=True)

preprocess_fail_count = 0

for class_name in CLASS_DIRS:
    src_dir = os.path.join(SOURCE_ROOT, class_name)
    pre_dir = os.path.join(PREPROCESSED_ROOT, class_name)

    if not os.path.isdir(src_dir):
        print(f"[경고] 폴더를 찾을 수 없음: {src_dir}")
        continue

    wav_files = [f for f in os.listdir(src_dir) if f.lower().endswith(".wav")]
    print(f"[{class_name}] 전처리 시작 ({len(wav_files)}개)")

    for fname in wav_files:
        src_path = os.path.join(src_dir, fname)
        try:
            y, _ = librosa.load(src_path, sr=SAMPLE_RATE, mono=True)
            y_clean = preprocess_audio(y, SAMPLE_RATE)
            sf.write(os.path.join(pre_dir, fname), y_clean, SAMPLE_RATE)
        except Exception as e:
            print(f"  [에러] {fname} 전처리 실패: {e}")
            preprocess_fail_count += 1

    print(f"[{class_name}] 전처리 완료")

print(f"\n전처리 실패 파일 수: {preprocess_fail_count}")
print("전처리된 결과는 preprocessed_dataset 폴더에 저장되었습니다.")

[0_indoor_alarms] 전처리 시작 (37개)


/tmp/ipykernel_481/2837029377.py:22: UserWarning: PySoundFile failed. Trying audioread instead.
  y, _ = librosa.load(src_path, sr=SAMPLE_RATE, mono=True)
/usr/local/lib/python3.12/dist-packages/librosa/core/audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)


[0_indoor_alarms] 전처리 완료
[1_outdoor_warnings] 전처리 시작 (24개)
[1_outdoor_warnings] 전처리 완료
[2_emergency_alarms] 전처리 시작 (29개)
[2_emergency_alarms] 전처리 완료

전처리 실패 파일 수: 0
전처리된 결과는 preprocessed_dataset 폴더에 저장되었습니다.


In [6]:
import IPython.display as ipd

check_class = CLASS_DIRS[0]
check_dir_orig = os.path.join(SOURCE_ROOT, check_class)
check_dir_pre = os.path.join(PREPROCESSED_ROOT, check_class)

orig_files = [f for f in os.listdir(check_dir_orig) if f.lower().endswith(".wav")]
if orig_files:
    sample = orig_files[0]
    print("전처리 전:", sample)
    display(ipd.Audio(os.path.join(check_dir_orig, sample)))
    print("전처리 후:", sample)
    display(ipd.Audio(os.path.join(check_dir_pre, sample)))

전처리 전: 벨소리2.wav


전처리 후: 벨소리2.wav


In [7]:
def load_audio(path, sr=SAMPLE_RATE):
    y, _ = librosa.load(path, sr=sr, mono=True)
    return y

def time_shift(y, sr=SAMPLE_RATE, shift_max=0.3):
    """오디오를 시간축으로 좌우 이동 (빈 공간은 0으로 채움)"""
    shift_amt = int(sr * random.uniform(-shift_max, shift_max))
    return np.roll(y, shift_amt)

def pitch_shift(y, sr=SAMPLE_RATE, n_steps_range=(-3, 3)):
    """음높이를 랜덤하게 변경 (반음 단위)"""
    n_steps = random.uniform(*n_steps_range)
    return librosa.effects.pitch_shift(y, sr=sr, n_steps=n_steps)

def time_stretch(y, rate_range=(0.85, 1.15)):
    """재생 속도를 랜덤하게 변경 (0.85배~1.15배)"""
    rate = random.uniform(*rate_range)
    return librosa.effects.time_stretch(y, rate=rate)

def add_noise(y, noise_factor_range=(0.002, 0.01)):
    """가우시안 화이트 노이즈 추가"""
    noise_factor = random.uniform(*noise_factor_range)
    noise = np.random.randn(len(y))
    return y + noise_factor * noise

def change_volume(y, gain_range=(0.6, 1.4)):
    """볼륨(게인)을 랜덤하게 조절"""
    gain = random.uniform(*gain_range)
    y_new = y * gain
    max_val = np.max(np.abs(y_new))
    if max_val > 1.0:
        y_new = y_new / max_val
    return y_new

def time_mask(y, sr=SAMPLE_RATE, mask_max_ratio=0.1):
    """오디오 일부 구간을 무음으로 마스킹 (occlusion에 대한 강건성 확보)"""
    y_new = y.copy()
    mask_len = int(len(y) * random.uniform(0.02, mask_max_ratio))
    if mask_len <= 0 or len(y) <= mask_len:
        return y_new
    start = random.randint(0, len(y) - mask_len)
    y_new[start:start + mask_len] = 0.0
    return y_new

AUGMENTATIONS = [
    ("shift", time_shift),
    ("pitch", pitch_shift),
    ("stretch", time_stretch),
    ("noise", add_noise),
    ("volume", change_volume),
    ("mask", time_mask),
]

def normalize_audio(y):
    """클리핑 방지를 위한 안전한 정규화"""
    max_val = np.max(np.abs(y))
    if max_val > 0:
        y = y / max_val * 0.95
    return y

In [8]:
total_original = 0
total_generated = 0

for class_name in CLASS_DIRS:
    src_dir = os.path.join(PREPROCESSED_ROOT, class_name)   # 전처리된 깨끗한 wav를 입력으로 사용
    dst_dir = os.path.join(OUTPUT_ROOT, class_name)

    if not os.path.isdir(src_dir):
        print(f"[경고] 폴더를 찾을 수 없음: {src_dir}")
        continue

    wav_files = [f for f in os.listdir(src_dir) if f.lower().endswith(".wav")]
    print(f"\n[{class_name}] 원본 {len(wav_files)}개 파일 처리 시작")

    for fname in wav_files:
        src_path = os.path.join(src_dir, fname)
        base_name = os.path.splitext(fname)[0]

        try:
            y = load_audio(src_path)
        except Exception as e:
            print(f"  [에러] {fname} 로드 실패: {e}")
            continue

        shutil.copy(src_path, os.path.join(dst_dir, fname))
        total_original += 1

        for aug_name, aug_func in AUGMENTATIONS:
            try:
                y_aug = aug_func(y)
                y_aug = normalize_audio(y_aug)
                out_name = f"{base_name}_{aug_name}.wav"
                out_path = os.path.join(dst_dir, out_name)
                sf.write(out_path, y_aug, SAMPLE_RATE)
                total_generated += 1
            except Exception as e:
                print(f"  [에러] {fname} -> {aug_name} 증강 실패: {e}")

    print(f"[{class_name}] 완료")

print(f"\n원본 파일 수: {total_original}")
print(f"증강으로 생성된 파일 수: {total_generated}")
print(f"전체 파일 수(원본+증강): {total_original + total_generated}")


[0_indoor_alarms] 원본 37개 파일 처리 시작
[0_indoor_alarms] 완료

[1_outdoor_warnings] 원본 24개 파일 처리 시작
[1_outdoor_warnings] 완료

[2_emergency_alarms] 원본 29개 파일 처리 시작
[2_emergency_alarms] 완료

원본 파일 수: 90
증강으로 생성된 파일 수: 540
전체 파일 수(원본+증강): 630


In [9]:
shutil.make_archive("augmented_dataset", "zip", OUTPUT_ROOT)

from google.colab import files
files.download("augmented_dataset.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>